# Replication notebook
This notebook walks through the package step by step: data loading, feature construction, sample splitting, model estimation, and evaluation.

In [ ]:
from data_crsp_compustat import DataConfig, clean_panel, impute_missing_predictors
from features import FeatureConfig, build_feature_panel, prepare_split_matrices
from models_linear import fit_pooled_ols, tune_huber_regression, tune_elastic_net, tune_pcr, tune_pls
from models_glm_spline import tune_glm_spline
from models_trees import tune_decision_tree, tune_gbrt, tune_random_forest
from models_mlp import tune_mlp_models
from evaluation import evaluate_all_models

In [ ]:
data_config = DataConfig(
    crsp_path='path/to/crsp_monthly.csv',
    compustat_path='path/to/compustat_fundamentals.csv',
    link_path='path/to/ccm_link.csv',
    rf_path='path/to/risk_free_monthly.csv',
)
feature_config = FeatureConfig(include_macro_interactions=False)

In [ ]:
panel = clean_panel(data_config)
feature_panel, feature_cols = build_feature_panel(panel, feature_config)
feature_panel = impute_missing_predictors(feature_panel, feature_cols, data_config)
split = prepare_split_matrices(feature_panel, feature_cols, train_end='1986-12-31', val_end='2003-12-31', test_end='2016-12-31', scale=True)
feature_panel.head()

In [ ]:
X_train, y_train = split['X_train_scaled'], split['y_train']
X_val, y_val = split['X_val_scaled'], split['y_val']
X_test, y_test = split['X_test_scaled'], split['y_test']
benchmark_features = [c for c in ['log_me', 'bm', 'mom_12_1'] if c in X_train.columns]

In [ ]:
models = {}
models['OLS_benchmark'] = fit_pooled_ols(X_train[benchmark_features], y_train)
models['Huber'] = tune_huber_regression(X_train, y_train, X_val, y_val)['model']
models['ElasticNet'] = tune_elastic_net(X_train, y_train, X_val, y_val)['model']
models['PCR'] = tune_pcr(X_train, y_train, X_val, y_val)['model']
models['PLS'] = tune_pls(X_train, y_train, X_val, y_val)['model']
models['GLM_spline'] = tune_glm_spline(X_train, y_train, X_val, y_val).model
models['MLP'] = tune_mlp_models(X_train, y_train, X_val, y_val)['model']

In [ ]:
tree_models = {}
tree_models['DecisionTree'] = tune_decision_tree(split['X_train'], split['y_train'], split['X_val'], split['y_val'])['model']
tree_models['GBRT'] = tune_gbrt(split['X_train'], split['y_train'], split['X_val'], split['y_val'])['model']
tree_models['RandomForest'] = tune_random_forest(split['X_train'], split['y_train'], split['X_val'], split['y_val'])['model']

In [ ]:
results_scaled = evaluate_all_models(models, {'X_test': X_test, 'y_test': y_test, 'meta_test': split['meta_test']})
results_raw = evaluate_all_models(tree_models, {'X_test': split['X_test'], 'y_test': split['y_test'], 'meta_test': split['meta_test']})
results_scaled['stock_level'], results_raw['stock_level']